<a href="https://colab.research.google.com/github/ashitasingh1230-commits/IT_support_ticket_analysis/blob/main/skincare_sales_SQL_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Load and inspect the *data*

In [26]:
import pandas as pd
df=pd.read_csv('cosmetics_sales_data.csv')
df.head()

,Sales Person,Country,Product,Date,Amount ($),Boxes Shipped
0,Lucas Verma,Canada,Aloe Vera Gel,2022-04-30,7897.13,358
1,Ethan Reddy,UK,Aloe Vera Gel,2022-01-25,16376.88,449
2,Ananya Gupta,India,Body Butter Cream,2022-08-22,5599.68,264
3,Ananya Gupta,New Zealand,Salicylic Acid Cleanser,2022-08-26,2966.47,144
4,Sophia Nair,UK,Body Butter Cream,2022-05-19,6828.68,484


# Check the size of the dataset

In [15]:
df.shape

(374, 6)

# Check for missing values and data types

In [16]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 374 entries, 0 to 373
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Sales Person   374 non-null    object 
 1   Country        374 non-null    object 
 2   Product        374 non-null    object 
 3   Date           374 non-null    object 
 4   Amount ($)     374 non-null    float64
 5   Boxes Shipped  374 non-null    int64  
dtypes: float64(1), int64(1), object(4)
memory usage: 17.7+ KB


**Import SQLite and create a database**





In [17]:
import sqlite3
conn=sqlite3.connect('skincare_sales.db')
df.to_sql('sales',conn, if_exists='replace',index=False)

374

Load And Inspect The Data through SQL

In [18]:
query= """
SELECT * from sales LIMIT 5
"""
pd.read_sql(query,conn)

,Sales Person,Country,Product,Date,Amount ($),Boxes Shipped
0,Lucas Verma,Canada,Aloe Vera Gel,2022-04-30,7897.13,358
1,Ethan Reddy,UK,Aloe Vera Gel,2022-01-25,16376.88,449
2,Ananya Gupta,India,Body Butter Cream,2022-08-22,5599.68,264
3,Ananya Gupta,New Zealand,Salicylic Acid Cleanser,2022-08-26,2966.47,144
4,Sophia Nair,UK,Body Butter Cream,2022-05-19,6828.68,484


Select specific columns

In [19]:
query= """
Select "Product", "Amount ($)"
from sales
LIMIT 5
"""
pd.read_sql(query,conn)

,Product,Amount ($)
0,Aloe Vera Gel,7897.13
1,Aloe Vera Gel,16376.88
2,Body Butter Cream,5599.68
3,Salicylic Acid Cleanser,2966.47
4,Body Butter Cream,6828.68


# **Total revenue by product**

In [20]:
query = """
select "Product" , sum("Amount ($)") as total_revenue
from sales
group by Product
order by total_revenue desc
"""
pd.read_sql(query,conn)

,Product,total_revenue
0,Tea Tree Moisturizer,260905.44
1,Hydrating Face Serum,250323.33
2,Hair Repair Oil,232864.77
3,Anti-Aging Serum,232248.00
4,Body Butter Cream,222923.58
5,SPF 50 Sunscreen,218576.97
6,Vitamin C Cream,218210.84
7,Aloe Vera Gel,202901.75
8,Face Sheet Masks,197882.89
9,Under Eye Cream,195936.57


# Total revenue by **country**

In [21]:
query = """
select "Country" , round(sum("Amount ($)"),2) as total_revenue
from sales
group by Country
order by total_revenue desc
"""
pd.read_sql(query,conn)

,Country,total_revenue
0,USA,628487.86
1,New Zealand,557059.85
2,Australia,505497.64
3,UK,497061.54
4,Canada,374562.31
5,India,346434.92


# The USA generated the highest total revenue ($628K), nearly 1.8x more than India, the lowest-performing country ($346K) — while New Zealand, Australia, and the UK formed a fairly tight middle tier ($497K-$557K).

# Revenue by country AND product (top combinations only)

In [22]:
query = """
select "Country" , "Product" , round(sum("Amount ($)"),2) as total_revenue
from sales
group by Country,Product
order by total_revenue desc
LIMIT 10
"""
pd.read_sql(query,conn)

,Country,Product,total_revenue
0,USA,Anti-Aging Serum,113821.81
1,Australia,Hair Repair Oil,91002.87
2,New Zealand,SPF 50 Sunscreen,87897.03
3,New Zealand,Body Butter Cream,86730.86
4,UK,Hydrating Face Serum,75719.24
5,UK,Vitamin C Cream,72516.17
6,Australia,Tea Tree Moisturizer,65869.40
7,USA,Rose Water Toner,65113.68
8,USA,Niacinamide Toner,63166.53
9,New Zealand,Under Eye Cream,60475.28


USA's top-selling product by revenue was Anti-Aging Serum ($113,822), not Tea Tree Moisturizer — the overall best-selling product globally. This suggests demand varies meaningfully by country, and a one-size-fits-all product strategy would miss country-specific preferences.

# **Extract month from the Date column**

In [23]:
query = """
select "Date" from sales limit 5
"""
pd.read_sql(query,conn)

,Date
0,2022-04-30
1,2022-01-25
2,2022-08-22
3,2022-08-26
4,2022-05-19


In [24]:
query = """
select strftime('%Y-%m',"Date") as Month ,round(sum("Amount ($)"),2) as total_revenue
from sales
group by Month
order by Month
"""
pd.read_sql(query,conn)

,Month,total_revenue
0,2022-01,359762.51
1,2022-02,214024.56
2,2022-03,484101.59
3,2022-04,452650.04
4,2022-05,396609.09
5,2022-06,367001.65
6,2022-07,359655.73
7,2022-08,275298.95


# Revenue peaked in March 2022 ($484K) after a dip in February ($214K), then gradually declined each month through August ($275K). Note: the dataset only spans January-August 2022, so this reflects a partial-year trend, not a full annual cycle.

# **Total revenue by sales person**

In [29]:
query= """
select "Sales Person" , round(sum("Amount ($)"),2) as total_revenue , count(*) as num_sales
from sales
group by "Sales Person"
order by total_revenue desc
"""
pd.read_sql(query,conn)


,Sales Person,total_revenue,num_sales
0,Olivia D'Souza,387405.91,47
1,Sophia Nair,319887.82,40
2,Isabella Roy,302087.60,38
3,Ethan Reddy,298595.61,35
4,Lucas Verma,295166.91,36
5,Ananya Gupta,293204.67,42
6,Noah Mehta,272188.08,40
7,Liam Patel,270960.55,36
8,Ava Sharma,246174.28,28
9,Mason Kapoor,223432.69,32


# **Average revenue per sale, by salesperson**

In [32]:
query = """
select "Sales Person",
round(sum("Amount ($)"),2) as total_revenue ,
count(*) as num_sales,
round(sum("Amount ($)")/ count(*) ,2) as avg_revenue_sale
from sales
group by "Sales Person"
order by avg_revenue_sale desc
"""
pd.read_sql(query,conn)

,Sales Person,total_revenue,num_sales,avg_revenue_sale
0,Ava Sharma,246174.28,28,8791.94
1,Ethan Reddy,298595.61,35,8531.30
2,Olivia D'Souza,387405.91,47,8242.68
3,Lucas Verma,295166.91,36,8199.08
4,Sophia Nair,319887.82,40,7997.20
5,Isabella Roy,302087.60,38,7949.67
6,Liam Patel,270960.55,36,7526.68
7,Mason Kapoor,223432.69,32,6982.27
8,Ananya Gupta,293204.67,42,6981.06
9,Noah Mehta,272188.08,40,6804.70


## Ranking salespeople by total revenue versus average deal size tells two different stories: Olivia D'Souza leads in total revenue($387K)through higher sales volume (47 transactions), while Ava Sharma has the highest average deal size ($8,792) despite fewer total sales (28) — suggesting Ava tends to close larger individual deals, while Olivia sells more frequently. A sales strategy shouldn't treat these two patterns the same way.

# **Export a summary table from SQL to CSV**

In [34]:
query= """
select "Product" , round(sum("Amount ($)"),2) as total_revenue
from sales
group by "Product"
order by total_revenue desc
"""
result=pd.read_sql(query,conn)
result.to_csv('revenue_by_product.csv',index=False)
print('Saved!')

Saved!
